In [2]:
import sys
!{sys.executable} -m pip install nbformat>=4.2.0 ipywidgets scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# EXP_009d0: Reproducibility Gate

## Purpose

This is **not** a new experiment. It is an exact replication of EXP_009aFIX under identical conditions.

## Hypothesis

**H0:** The Lucier Resonance results are deterministic. Running the same model, same prompts, same parameters, and same iteration schedule will produce identical terminal attractors and dissolution trajectories.

## Pass Criteria

1. All five prompts reach the same terminal tokens as the original run (`prolet` × 4, `Divine` × 1)
2. Cross-prompt cosine similarity matrix matches the original within ±0.01
3. Dissolution phase sequence is identical at all snapshot iterations

## Gate Condition

> **If this test fails, Stages 1–3 cannot proceed.** Non-reproducibility would indicate that the observed attractors are sensitive to floating-point noise, CUDA non-determinism, or other stochastic factors. This must be ruled out before any further interpretation.

---


In [3]:
# ============================================================
# STEP 1: CALIBRATION — Load the Organism
# ============================================================
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Running on: {device}")
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Running on: cpu
Architecture: 12 layers, 12 heads, d_model=768


In [4]:
# ============================================================
# STEP 2: CONFIGURATION — Identical to EXP_009aFIX
# ============================================================

ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100, 250, 500]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

# IDENTICAL prompt library from EXP_009aFIX
PROMPT_LIBRARY = {
    "Lucier":     "Am I sitting in a room different from the one you are in now",
    "Semantic":   "The Eiffel Tower is located in the city of",
    "Syntactic":  "The cat sat on the mat and then the",
    "Nonsense":   "Flurb glex morp wintly skade",
    "Imperative": "Calculate the sum of all prime numbers below",
}

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Room: Layers {LAYER_START} → {LAYER_END}")
print(f"Prompts: {list(PROMPT_LIBRARY.keys())}")
print(f"MODE: REPRODUCIBILITY TEST — identical to EXP_009aFIX")

Schedule: [0, 2, 3, 5, 10, 20, 50, 100, 250, 500]
Room: Layers 0 → 11
Prompts: ['Lucier', 'Semantic', 'Syntactic', 'Nonsense', 'Imperative']
MODE: REPRODUCIBILITY TEST — identical to EXP_009aFIX


In [5]:
# ============================================================
# STEP 3: THE CORE ENGINE — Identical to EXP_009aFIX
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies the Final LayerNorm before unembedding for correct decoding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter, schedule):
    """
    TOTAL Lucier Loop: iteratively re-inject the ENTIRE residual stream
    tensor (all token positions) through the layer slice.
    
    Returns a list of snapshot dicts at each scheduled iteration.
    """
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    # === Iteration 0: The original recording ===
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    initial_norm = current_tensor.norm().item()
    print(f"  Sequence length: {seq_len} tokens, initial norm: {initial_norm:.2f}")
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        # Also decode ALL positions for sentence reconstruction
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    # === Iterations 1 → max_iter: The TOTAL feedback loop ===
    for i in range(1, max_iter + 1):
        # Normalise to maintain energy level
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            
            # Decode all positions for sentence reconstruction
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  Snapshot @ iter {i:>3}: norm={current_tensor.norm().item():.2f}, "
                  f"cos_last={cos_sim_last:.6f}, cos_mean={cos_sim_mean:.6f}, "
                  f"pos_collapse={position_similarity:.4f}, "
                  f"top='{top_tokens_last[0][0]}'")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Total Resonance engine loaded.")

Total Resonance engine loaded.


In [6]:
# ============================================================
# STEP 4: RUN THE EXPERIMENT
# ============================================================

all_results = {}

for label, prompt in PROMPT_LIBRARY.items():
    print(f"\n{'='*60}")
    print(f"RECORDING: '{label}' — \"{prompt}\"")
    print(f"{'='*60}")
    
    snapshots = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    all_results[label] = snapshots
    print(f"  ✓ {len(snapshots)} snapshots captured.")

print(f"\n{'='*60}")
print(f"ALL RECORDINGS COMPLETE.")


RECORDING: 'Lucier' — "Am I sitting in a room different from the one you are in now"
  Sequence length: 15 tokens, initial norm: 1645.70
  Snapshot @ iter   2: norm=5855.82, cos_last=0.950058, cos_mean=0.694011, pos_collapse=0.9392, top=' the'
  Snapshot @ iter   3: norm=4605.57, cos_last=-0.070779, cos_mean=-0.698494, pos_collapse=0.8642, top=' the'
  Snapshot @ iter   5: norm=5115.17, cos_last=-0.375822, cos_mean=-0.312655, pos_collapse=0.9943, top=' Fem'
  Snapshot @ iter  10: norm=6119.48, cos_last=0.571521, cos_mean=0.586292, pos_collapse=0.9995, top=' capit'
  Snapshot @ iter  20: norm=5939.37, cos_last=0.883806, cos_mean=0.883844, pos_collapse=1.0000, top='.'
  Snapshot @ iter  50: norm=6040.40, cos_last=0.942395, cos_mean=0.942395, pos_collapse=1.0000, top=' Rousse'
  Snapshot @ iter 100: norm=5869.00, cos_last=0.999999, cos_mean=0.999999, pos_collapse=1.0000, top=' prolet'
  Snapshot @ iter 250: norm=5879.62, cos_last=1.000000, cos_mean=1.000000, pos_collapse=1.0000, top=' pr

---
## 5. Reproducibility Validation

### 5a. Sentence Dissolution Tables
Compare directly against EXP_009aFIX results.

In [7]:
# ============================================================
# VIS 5a: SENTENCE DISSOLUTION — Full position reconstruction
# ============================================================

for label in PROMPT_LIBRARY.keys():
    snapshots = all_results[label]
    md = f"### {label}: *\"{PROMPT_LIBRARY[label]}\"*\n\n"
    md += "| Iter | Reconstructed Output |\n"
    md += "|:---|:---|\n"
    for s in snapshots:
        tokens = s['all_position_tokens']
        clean = [t.replace('\n', '↵').replace('|', '\\|') for t in tokens]
        sentence = ' '.join(clean)
        md += f"| {s['iteration']} | {sentence} |\n"
    md += "\n"
    display(Markdown(md))

### Lucier: *"Am I sitting in a room different from the one you are in now"*

| Iter | Reconstructed Output |
|:---|:---|
| 0 | ↵ anda  the  on  a  room  with  from  the  one  I 're  in ? ? |
| 2 | ash ash ↵ ↵ ↵ ↵  the  the ↵ .  the . ↵  the  the |
| 3 | The ↵ ↵  the ↵ ↵  the  the  the .  the  the  the  the  the |
| 5 | ash ash ash ash  Canad  Canad  Canad  Canad  Canad  Canad  Canad  Canad  Canad  Fem  Fem |
| 10 |  FT  FT  FT  FT  FT  capit  capit  capit  capit  capit  capit  capit  capit  capit  capit |
| 20 | . . . . . . . . . . . . . . . |
| 50 |  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse |
| 100 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |
| 250 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |
| 500 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |



### Semantic: *"The Eiffel Tower is located in the city of"*

| Iter | Reconstructed Output |
|:---|:---|
| 0 | ↵  first - el  Tower  is  a  in  the  heart  of  London |
| 2 | ash ↵ ↵  the  the ↵ ↵  the ↵ ↵  the ↵ |
| 3 | ash ↵ ↵  the . ↵ ↵  the ↵ ↵  the ↵ |
| 5 | ash  Canad  Canad  Canad  Fem  Canad  Fem  Fem ↵  Canad  Event ↵ |
| 10 |  FT  FT  FT  FT  FT  FT  Ag  Ag  Ag  Ag  Ag  Ag |
| 20 |  injustice  injustice  injustice  injustice  injustice  injustice  injustice  injustice  injustice  injustice  injustice  injustice |
| 50 |  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse |
| 100 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |
| 250 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |
| 500 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |



### Syntactic: *"The cat sat on the mat and then the"*

| Iter | Reconstructed Output |
|:---|:---|
| 0 | ↵  first  was  on  the  floor ,  looked  looked  cat |
| 2 | ↵ ↵ ↵  the  the ↵  the  the  the . |
| 3 | The ↵ ↵  the  the .  the  the  the ↵ |
| 5 | ash  Canad  Canad  Canad  Canad  Canad  Canad  Canad  Canad  Fem |
| 10 |  Ag  Ag  Ag  Ag  Ag  Ag  Ag  Ag  Ag  Ag |
| 20 |  Zero  Zero  Zero  Zero  Zero  Zero  Zero  Zero  Zero  Zero |
| 50 |  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine |
| 100 |  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine |
| 250 |  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine |
| 500 |  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine  Divine |



### Nonsense: *"Flurb glex morp wintly skade"*

| Iter | Reconstructed Output |
|:---|:---|
| 0 | ↵ oyd  a ags ↵ hing / ry , ips , |
| 2 | ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ |
| 3 | The ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ |
| 5 | ash ash ash ash ash ash ash ash ash ash ash |
| 10 |  tem  Ag  Ag  Ag  Ag  Ag  Ag  Ag  Ag  Ag  Ag |
| 20 |  Difference  Difference  Difference  Difference  Difference  Difference  Difference  Difference  Difference  Difference  Difference |
| 50 |  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse |
| 100 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |
| 250 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |
| 500 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |



### Imperative: *"Calculate the sum of all prime numbers below"*

| Iter | Reconstructed Output |
|:---|:---|
| 0 | ↵ gary ate  the  number  of  the  the  numbers  in . |
| 2 | ↵ ↵ ↵ ↵ ↵  the ↵ ↵ ↵ ↵ ↵ |
| 3 | The ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ ↵ |
| 5 | ash  Fem minus  Fem  Fem  Fem  Fem ↵ ↵ ↵ ↵ |
| 10 |  FT  capit  capit  capit  capit  capit  capit  capit  capit  capit  capit |
| 20 | . . . . . . . . . . . |
| 50 |  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse  Rousse |
| 100 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |
| 250 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |
| 500 |  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet  prolet |



In [8]:
# ============================================================
# VIS 5b: CROSS-PROMPT CONVERGENCE MATRIX
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="Viridis",
    title="Reproducibility Test: Cross-Prompt Cosine Similarity of Final States",
    text_auto=".3f",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=500)
fig_sim.show()

off_diag = sim_matrix[np.triu_indices(n, k=1)]
print(f"\nMean cross-prompt similarity: {off_diag.mean():.4f}")
print(f"Min cross-prompt similarity:  {off_diag.min():.4f}")
print(f"Max cross-prompt similarity:  {off_diag.max():.4f}")


Mean cross-prompt similarity: 0.8921
Min cross-prompt similarity:  0.7275
Max cross-prompt similarity:  1.0000


### 5c. Gate Decision

In [9]:
# ============================================================
# GATE DECISION: Compare against EXP_009aFIX expected results
# ============================================================

# Expected terminal tokens from EXP_009aFIX
EXPECTED_TERMINALS = {
    "Lucier": "prolet",
    "Semantic": "prolet",
    "Syntactic": "Divine",
    "Nonsense": "prolet",
    "Imperative": "prolet",
}

# Expected cross-prompt similarity from EXP_009aFIX
EXPECTED_MEAN_SIM = 0.892  # ±0.01

print("=" * 60)
print("REPRODUCIBILITY GATE ASSESSMENT")
print("=" * 60)

all_pass = True

# Check 1: Terminal tokens
print("\n--- Check 1: Terminal Token Match ---")
for label, expected in EXPECTED_TERMINALS.items():
    actual = all_results[label][-1]["top_tokens"][0][0].strip()
    match = expected in actual or actual in expected
    status = "✓ PASS" if match else "✗ FAIL"
    if not match:
        all_pass = False
    print(f"  {label}: expected '{expected}', got '{actual}' — {status}")

# Check 2: Cross-prompt similarity
print("\n--- Check 2: Cross-Prompt Similarity ---")
actual_mean_sim = off_diag.mean()
sim_match = abs(actual_mean_sim - EXPECTED_MEAN_SIM) < 0.02
status = "✓ PASS" if sim_match else "✗ FAIL"
if not sim_match:
    all_pass = False
print(f"  Expected ~{EXPECTED_MEAN_SIM:.3f}, got {actual_mean_sim:.4f} — {status}")

# Verdict
print(f"\n{'='*60}")
if all_pass:
    print("✓ GATE PASSED: Results are reproducible.")
    print("  Proceed to EXP_009d1 (Attractor Dominance).")
else:
    print("✗ GATE FAILED: Results are NOT reproducible.")
    print("  DO NOT proceed to Stages 1-3.")
    print("  Investigate sources of non-determinism.")
print(f"{'='*60}")

REPRODUCIBILITY GATE ASSESSMENT

--- Check 1: Terminal Token Match ---
  Lucier: expected 'prolet', got 'prolet' — ✓ PASS
  Semantic: expected 'prolet', got 'prolet' — ✓ PASS
  Syntactic: expected 'Divine', got 'Divine' — ✓ PASS
  Nonsense: expected 'prolet', got 'prolet' — ✓ PASS
  Imperative: expected 'prolet', got 'prolet' — ✓ PASS

--- Check 2: Cross-Prompt Similarity ---
  Expected ~0.892, got 0.8921 — ✓ PASS

✓ GATE PASSED: Results are reproducible.
  Proceed to EXP_009d1 (Attractor Dominance).


In [10]:
# ============================================================
# STEP 6: SAVE ARTIFACTS
# ============================================================
import os

save_dir = os.path.join("..", "_DATA", "EXP_009")
os.makedirs(save_dir, exist_ok=True)

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
        "all_position_tokens": [s["all_position_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(save_dir, "009d0_reproducibility_results.pt"))
print(f"[SAVED] {save_dir}/009d0_reproducibility_results.pt")

[SAVED] ..\_DATA\EXP_009/009d0_reproducibility_results.pt
